# Underwater Waste Detection — YOLO-NAS Training

YOLO-NAS (Neural Architecture Search) uses an automated search process to find the optimal
architecture for the accuracy/latency tradeoff — making it particularly relevant for deployment
on AUVs with limited compute.

**Variants trained here:** `yolo_nas_s` (fast) and optionally `yolo_nas_l` (accurate)

**Library:** `super-gradients` by Deci AI

In [ ]:
# ── Cell 1: GPU check & install ───────────────────────────────────────────
!nvidia-smi
!pip install -q super-gradients==3.7.1 roboflow

In [ ]:
# ── Cell 2: Mount Drive & paths ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

DRIVE_BASE       = Path('/content/drive/MyDrive/underwater_waste_detection')
YOLO_DATASET_DIR = str(DRIVE_BASE / 'trashcan_yolo')
CHECKPOINT_DIR   = str(DRIVE_BASE / 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

REPO_DIR = Path('/content/underwater-waste-detection')
if not REPO_DIR.exists():
    !git clone https://github.com/omprxkash/underwater-waste-detection.git {REPO_DIR}
os.chdir(REPO_DIR)

print(f'Dataset: {YOLO_DATASET_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────
MODEL_VARIANT  = 'yolo_nas_s'   # 'yolo_nas_s', 'yolo_nas_m', 'yolo_nas_l'
BATCH_SIZE     = 8
MAX_EPOCHS     = 100
INITIAL_LR     = 5e-4
NUM_CLASSES    = 3
CLASS_NAMES    = ['trash', 'bio', 'rov']
EXPERIMENT     = f'{MODEL_VARIANT}_underwater'

In [ ]:
# ── Cell 4: Verify dataset structure ─────────────────────────────────────
for split in ['train', 'val']:
    n_imgs = len(list(Path(YOLO_DATASET_DIR, 'images', split).glob('*')))
    n_lbls = len(list(Path(YOLO_DATASET_DIR, 'labels', split).glob('*.txt')))
    print(f'{split:6s}: {n_imgs:5d} images, {n_lbls:5d} labels')

In [ ]:
# ── Cell 5: Build data loaders ────────────────────────────────────────────
from super_gradients.training.dataloaders.dataloaders import (
    coco_detection_yolo_format_train,
    coco_detection_yolo_format_val,
)

shared = {'data_dir': YOLO_DATASET_DIR, 'classes': CLASS_NAMES}
dl_params = {'batch_size': BATCH_SIZE, 'num_workers': 2}

train_loader = coco_detection_yolo_format_train(
    dataset_params={**shared, 'images_dir': f'{YOLO_DATASET_DIR}/images/train',
                               'labels_dir': f'{YOLO_DATASET_DIR}/labels/train'},
    dataloader_params=dl_params,
)
val_loader = coco_detection_yolo_format_val(
    dataset_params={**shared, 'images_dir': f'{YOLO_DATASET_DIR}/images/val',
                               'labels_dir': f'{YOLO_DATASET_DIR}/labels/val'},
    dataloader_params=dl_params,
)
print('Data loaders ready.')

In [ ]:
# ── Cell 6: Load YOLO-NAS model ───────────────────────────────────────────
from super_gradients.training import models, Trainer

model = models.get(MODEL_VARIANT, num_classes=NUM_CLASSES)
trainer = Trainer(experiment_name=EXPERIMENT, ckpt_root_dir=CHECKPOINT_DIR)
print(f'Model loaded: {MODEL_VARIANT}')

In [ ]:
# ── Cell 7: Training parameters ───────────────────────────────────────────
from super_gradients.training.losses import PPYoloELoss
from super_gradients.training.metrics import DetectionMetrics_050
from super_gradients.training.models.detection_models.pp_yolo_e import PPYoloEPostPredictionCallback

train_params = {
    'silent_mode': False,
    'average_best_models': True,
    'warmup_mode': 'linear_epoch_step',
    'warmup_initial_lr': 1e-6,
    'lr_warmup_epochs': 3,
    'initial_lr': INITIAL_LR,
    'lr_mode': 'cosine',
    'cosine_final_lr_ratio': 0.1,
    'optimizer': 'Adam',
    'optimizer_params': {'weight_decay': 0.0001},
    'zero_weight_decay_on_bias_and_bn': True,
    'ema': True,
    'ema_params': {'decay': 0.9, 'decay_type': 'threshold'},
    'max_epochs': MAX_EPOCHS,
    'mixed_precision': True,
    'loss': PPYoloELoss(
        use_static_assigner=False,
        num_classes=NUM_CLASSES,
        reg_max=16,
    ),
    'valid_metrics_list': [
        DetectionMetrics_050(
            score_thres=0.1,
            top_k_predictions=300,
            num_cls=NUM_CLASSES,
            normalize_targets=True,
            post_prediction_callback=PPYoloEPostPredictionCallback(
                score_threshold=0.01,
                nms_top_k=1000,
                max_predictions=300,
                nms_threshold=0.7,
            ),
        )
    ],
    'metric_to_watch': 'mAP@0.50',
}

In [ ]:
# ── Cell 8: TensorBoard ───────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir {CHECKPOINT_DIR} --bind_all

In [ ]:
# ── Cell 9: Train ─────────────────────────────────────────────────────────
trainer.train(
    model=model,
    training_params=train_params,
    train_loader=train_loader,
    valid_loader=val_loader,
)
print(f'\nTraining complete. Checkpoints saved to: {CHECKPOINT_DIR}/{EXPERIMENT}')

In [ ]:
# ── Cell 10: Sample inference with trained YOLO-NAS ───────────────────────
import os

best_ckpt = os.path.join(CHECKPOINT_DIR, EXPERIMENT, 'ckpt_best.pth')

inference_model = models.get(
    MODEL_VARIANT,
    num_classes=NUM_CLASSES,
    checkpoint_path=best_ckpt,
)

# Pick a sample image from validation set
sample_images = list(Path(YOLO_DATASET_DIR, 'images', 'val').glob('*.jpg'))[:3]
for img_path in sample_images:
    predictions = inference_model.predict(str(img_path), conf=0.4)
    predictions.show()
    print(f'Predictions shown for: {img_path.name}')